In [0]:
use catalog olist_retail;
use schema gold_dataset;

-- TOP 3 PRODUCTS CATEGORY IN EACH STATE
create or replace table olist_retail.gold_dataset.top_3_products_in_state
with cte as (
    select
        c.customer_state as state,
        p.product_category_name,
        SUM(fs.payment_value) AS REVENUE
    from fact_sales fs
    left join dim_product p
        on fs.product_id = p.product_id
    left join dim_customer c
        on fs.customer_id = c.customer_id
    group by p.product_category_name, state
),
ranked_product AS(
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY STATE
            ORDER BY REVENUE DESC
        ) AS RN
    FROM CTE
)
   ( SELECT *
    FROM ranked_product
    WHERE rn <= 3
);

-- PRODUCT DEMAND ANALYSIS
CREATE OR REPLACE TABLE product_demand AS
SELECT
    product_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(payment_value) AS revenue
FROM fact_sales
GROUP BY product_id;

-- PRODUCT CHURN ANALYSIS
CREATE OR REPLACE TABLE product_churn AS
SELECT
    product_id,
    MAX(order_date) AS last_order_date,
    DATEDIFF(
        (SELECT MAX(order_date) FROM fact_sales),
             MAX(order_date)) AS days_since_last_sale
FROM fact_sales
GROUP BY product_id;

-- PRODUCT SEASONALITY ANALYSIS
CREATE or REPLACE TABLE product_seasonality AS
SELECT
    product_id,
    MONTH(order_date) AS month_no,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(payment_value) AS revenue
FROM fact_sales
GROUP BY
    product_id,
    MONTH(order_date);

-- PROFITABILITY ANALYSIS
CREATE or REPLACE TABLE product_profitability AS
SELECT
    product_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(price) AS price,
    SUM(payment_value) AS revenue,
    SUM(payment_value) - SUM(price) AS profit
FROM fact_sales
GROUP BY
    product_id;
